# JuaKazi Gender Bias Engine — Multilingual Corrector Training

**Version:** v1 (Jun 2026)  
**Scope:** All 6 languages — SW, HA, ZU, KI, FR, EN  
**Model:** `castorini/afriteva_v2_base` — AfriTeVa v2 seq2seq, pre-trained on African text  
**Task:** Seq2seq bias correction — biased sentence → neutral rewrite  
**Data:** `training_data_correction.csv` (~10K pairs, all languages)  
**Input prefix:** `correct bias {lang}: {biased sentence}`  
**Output:** corrected/neutral sentence  
**Push to:** `juakazike/multilingual-bias-corrector-v1`

## Hardware
Kaggle T4 x2 (recommended). ~2-3 hours for 5 epochs.

In [ ]:
# Cell A1: Install dependencies
import subprocess, sys

def install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

install([
    'transformers>=4.45.0',
    'datasets>=2.18.0',
    'accelerate>=0.34.0',
    'sentencepiece>=0.1.99',
    'evaluate>=0.4.0',
    'sacrebleu>=2.3.0',
    'huggingface_hub>=0.22.0',
])
print('Dependencies installed — restart runtime now, then run Cell A2')

In [ ]:
# Cell A2: Verify environment
import torch, transformers, datasets
print(f'PyTorch:       {torch.__version__}')
print(f'Transformers:  {transformers.__version__}')
print(f'Datasets:      {datasets.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU count:     {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# Cell A3: Load correction pairs
# On Kaggle: upload training_data_correction.csv as a dataset
# Dataset name: stellaoiro/juakazi-correction-data
# Path: /kaggle/input/juakazi-correction-data/training_data_correction.csv

import pandas as pd, numpy as np, os

DATA_PATH = '/kaggle/input/juakazi-correction-data/training_data_correction.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/training_data_correction.csv'

df = pd.read_csv(DATA_PATH)
print(f'Total pairs: {len(df):,}')
print(df['language'].value_counts())
print()
print('Sample:')
for lang in ['sw','ha','zu','ki','fr','en']:
    row = df[df['language']==lang].iloc[0]
    print(f"  [{lang.upper()}] IN:  {str(row['input_text'])[:80]}")
    print(f"  [{lang.upper()}] OUT: {str(row['target_text'])[:80]}")
    print()

In [ ]:
# Cell A4: Build seq2seq dataset with language prefix
# Input format: 'correct bias {lang}: {biased sentence}'
# Target:       corrected sentence

def make_prefix(row):
    return f"correct bias {row['language']}: {str(row['input_text']).strip()}"

df['model_input']  = df.apply(make_prefix, axis=1)
df['model_target'] = df['target_text'].astype(str).str.strip()

# Drop empty/identical pairs
df = df[df['model_input'].str.len() > 10]
df = df[df['model_target'].str.len() > 2]
df = df[df['model_input'] != df['model_target']]

print(f'Valid pairs after filtering: {len(df):,}')
print(f"By language: {df['language'].value_counts().to_dict()}")

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    df, test_size=0.1, random_state=42, stratify=df['language']
)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

In [ ]:
# Cell A5: Load AfriTeVa tokenizer and model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = 'castorini/afriteva_v2_base'

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Loading model: {MODEL_NAME}')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Vocab size:   {tokenizer.vocab_size:,}')

In [ ]:
# Cell A6: Tokenize
from datasets import Dataset

MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 128

def tokenize(batch):
    model_inputs = tokenizer(
        batch['model_input'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding='max_length',
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch['model_target'],
            max_length=MAX_TARGET_LEN,
            truncation=True,
            padding='max_length',
        )
    # Replace padding id in labels with -100 so loss ignores padding
    labels['input_ids'] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in label]
        for label in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_pandas(train_df[['model_input','model_target']].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[['model_input','model_target']].reset_index(drop=True))

train_tok = train_ds.map(tokenize, batched=True, batch_size=64, remove_columns=['model_input','model_target'])
val_tok   = val_ds.map(tokenize,   batched=True, batch_size=64, remove_columns=['model_input','model_target'])

train_tok.set_format('torch')
val_tok.set_format('torch')

print(f'Train tokens: {len(train_tok):,} | Val tokens: {len(val_tok):,}')

In [ ]:
# Cell A7: Train
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import evaluate, numpy as np

EPOCHS      = 5
BATCH_SIZE  = 16
LR          = 5e-5
OUTPUT_DIR  = '/kaggle/working/afriteva-corrector'

sacrebleu = evaluate.load('sacrebleu')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]
    result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)
    return {'bleu': round(result['score'], 2)}

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=100,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='bleu',
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    logging_steps=50,
    report_to='none',
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f'Training on {len(train_tok):,} pairs | {EPOCHS} epochs | batch {BATCH_SIZE} | lr {LR}')
trainer.train()

metrics = trainer.evaluate()
print('\n=== Final validation metrics ===')
for k, v in metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# Cell A8: Quick inference test
def correct(text, lang):
    prompt = f'correct bias {lang}: {text}'
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, num_beams=4, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

test_sentences = [
    ('sw', 'Daktari wa kiume alifika hospitalini asubuhi.'),
    ('ha', 'Likitan namiji ne kawai zai iya jagorantar asibiti.'),
    ('zu', 'Udokotela wesilisa weza esibhedlela ekuseni.'),
    ('ki', 'Mundũ wa mũrũme nĩwe ũngĩ gũtwara mũciĩ.'),
    ('fr', 'Le président a dirigé la réunion comme un vrai homme.'),
    ('en', 'The chairman will lead the board meeting.'),
]

model.eval()
print('=== Inference test ===')
for lang, sentence in test_sentences:
    result = correct(sentence, lang)
    print(f'[{lang.upper()}] IN:  {sentence}')
    print(f'[{lang.upper()}] OUT: {result}')
    print()

In [ ]:
# Cell A9: Push to HuggingFace
from huggingface_hub import HfApi, login
import os, json

HF_TOKEN   = os.environ.get('HF_TOKEN', '')
MODEL_REPO = 'juakazike/multilingual-bias-corrector-v1'

if not HF_TOKEN:
    print('Set HF_TOKEN in Kaggle secrets (Add-ons > Secrets > HF_TOKEN)')
else:
    login(token=HF_TOKEN)
    trainer.push_to_hub(MODEL_REPO)
    print(f'Model pushed to https://huggingface.co/{MODEL_REPO}')

    card = '''---
language:
- sw
- ha
- zu
- ki
- fr
- en
tags:
- gender-bias
- seq2seq
- correction
- african-nlp
license: apache-2.0
---

# JuaKazi Multilingual Bias Corrector v1

Seq2seq model for gender bias correction across 6 languages.
Fine-tuned from castorini/afriteva_v2_base on ~7K correction pairs.

## Input format
correct bias {lang}: {biased sentence}

Where lang is one of: sw, ha, zu, ki, fr, en
'''

    api = HfApi()
    api.upload_file(
        path_or_fileobj=card.encode(),
        path_in_repo='README.md',
        repo_id=MODEL_REPO,
    )
    print('Model card written.')

In [ ]:
# Cell A10: Log metrics to eval/metrics.json
import json, os

metrics_path = '/kaggle/working/metrics_update.json'
bleu_score = metrics.get('eval_bleu', 0.0)

update = {
    'corrector': {
        'model': 'juakazike/multilingual-bias-corrector-v1',
        'base': 'castorini/afriteva_v2_base',
        'val_bleu': bleu_score,
        'train_pairs': len(train_tok),
        'val_pairs': len(val_tok),
        'languages': ['sw','ha','zu','ki','fr','en'],
        'trained': '2026-06',
    }
}

with open(metrics_path, 'w') as f:
    json.dump(update, f, indent=2)

print(f'Metrics saved to {metrics_path}')
print(f'BLEU: {bleu_score}')
print('Download this file and merge into eval/metrics.json')